# 01 Run Orchestrator

이 노트북은 기존 `experiment.ipynb`를 기반으로 한 실행용 엔트리포인트입니다.


# Condition Shift Baseline Notebook

이 노트북은 `condition_shift_baseline` 실험을 실행하고 결과를 비교하기 위한 orchestrator다.

이 노트북에서 하는 일:

- Colab/서버에서 Git checkout과 dataset 준비 상태를 맞춘다.
- `PatchCore`, `UniVAD` runner를 `single` 또는 `sequential` 모드로 실행한다.
- 실행 결과로 남는 `summary.json`, `log.txt`, clean-eval artifact를 다시 읽는다.
- clean 재현성, shift 취약성, baseline 간 차이를 표와 heatmap으로 정리한다.

원칙:

- 실험 핵심 로직은 노트북 안이 아니라 versioned `.py` runner에 둔다.
- 노트북은 실행 흐름, 상태 확인, 결과 비교에 집중한다.
- 같은 노트북으로 `PatchCore -> UniVAD` 순차 실행과 사후 비교를 모두 처리한다.


## 경로 설정

Drive mount, repo root, report root 같은 공통 경로를 먼저 고정한다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name == 'ReGraM'), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']
CORE_SRC = EXP_ROOT / 'src' / 'core'
UNIVAD_SRC = EXP_ROOT / 'src' / 'univad'
for source_root in (CORE_SRC, UNIVAD_SRC):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

# Colab/Jupyter keeps imported modules alive after git pull. Reload notebook helpers
# so rerunning this cell picks up the current checkout without a full runtime restart.
for module_name in ("notebook_orchestration", "dashboard_loader", "setup_runtime"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from dashboard_loader import (
    display_dashboard_tables,
    display_wandb_panel_recommendations,
    load_dashboard_frames,
    render_visual_dashboard,
)
from notebook_orchestration import (
    SEVERITY_ORDER,
    build_baseline_specs,
    build_requested_run_configs,
    discover_manifest_names,
    display_environment_summary,
    display_experiment_preset,
    display_run_plan,
    display_title,
    resolve_manifest_paths,
    resolve_requested_baselines,
    run_execution_queue,
    validate_baseline_names,
)
from setup_runtime import default_univad_setup_flags, evaluate_baseline_readiness, run_baseline_setup


DEFAULT_PATCHCORE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
run_history = []

display_environment_summary(REPO_ROOT, EXP_ROOT, REPORT_ROOT)
print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Git Checkout

Colab 런타임에서는 먼저 repo를 clone 또는 pull 해서 notebook이 호출할 runner와 helper 코드를 최신 상태로 맞춘다.

이 셀 이후부터는 `/content/ReGraM` 기준 경로를 사용한다.


In [ ]:
REPO_URL = 'https://github.com/outSeop/ReGraM.git'
REPO_DIR = Path('/content/ReGraM')
git_cmd = (
    f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
    f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
)
print(git_cmd)
subprocess.run(['bash', '-lc', git_cmd], check=True)

REPO_ROOT = REPO_DIR.resolve()
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

print('updated REPO_ROOT =', REPO_ROOT)
print('updated EXP_ROOT =', EXP_ROOT)


## Dataset Load

LOCO raw dataset tar를 runtime으로 가져와서 runner가 바로 읽을 수 있는 위치에 푼다.


In [ ]:
from pathlib import Path
import shutil
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = Path("/content/ReGraM/data/row")
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"


def find_nested_dataset_root(base_dir: Path, dataset_name: str):
    direct = base_dir / dataset_name
    if direct.exists():
        return direct
    matches = [path for path in base_dir.rglob(dataset_name) if path.is_dir()]
    if not matches:
        return None
    matches = sorted(matches, key=lambda path: len(path.parts))
    return matches[0]


def normalize_dataset_root(base_dir: Path, dataset_name: str):
    target_root = base_dir / dataset_name
    discovered_root = find_nested_dataset_root(base_dir, dataset_name)
    if discovered_root is None:
        return target_root, False
    if discovered_root == target_root:
        return target_root, False
    print(f"normalize dataset root: {discovered_root} -> {target_root}")
    target_root.parent.mkdir(parents=True, exist_ok=True)
    if target_root.exists():
        raise RuntimeError(f"target dataset root already exists: {target_root}")
    shutil.move(str(discovered_root), str(target_root))
    return target_root, True


print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

runtime_root, normalized = normalize_dataset_root(runtime_row, "mvtec_loco_anomaly_detection")
print("normalized:", normalized)
print("done")
print(runtime_root.exists(), runtime_root)


## Optional Dataset Bootstrap

Git checkout 이후 prepared dataset이나 보조 asset이 더 필요하면 별도 bootstrap script를 호출한다.

역할 분리:

- 코드 동기화: Git checkout 셀
- 데이터/보조 자산 준비: bootstrap 셀


In [ ]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


## Experiment Control Center

이 아래 셀들은 baseline 선택, readiness 확인, 실행, 결과 비교를 하나의 실험 루프로 묶는다.

사용 흐름:

- config 셀에서 `RUN_MODE`, baseline 목록, category, manifest 범위를 정한다.
- setup 셀에서 baseline별 외부 repo, dependency, checkpoint, python module 상태를 점검한다.
- execution 셀에서 현재 queue를 순서대로 실행한다.
- result dashboard에서 누적 summary를 읽어 clean/shift 결과를 비교한다.


## Runner Config

여기서 이번 실행의 범위를 결정한다.

- `RUN_MODE = 'single'`: 특정 baseline 하나만 실행
- `RUN_MODE = 'sequential'`: `SEQUENTIAL_BASELINES` 순서대로 연속 실행
- `DASHBOARD_BASELINES`: 결과 대시보드가 읽을 baseline 목록
- `CATEGORIES`, `MANIFEST_NAMES`: 이번 실험 입력 범위


In [ ]:
RUN_MODE = 'sequential'  # choose 'single' or 'sequential'
RUN_BASELINE = 'PatchCore'      # used only when RUN_MODE == 'single'
SEQUENTIAL_BASELINES = ['UniVAD']
DASHBOARD_BASELINES = ['UniVAD']
CATEGORIES = ['breakfast_box']
AUTO_DISCOVER_MANIFESTS = True
MANIFEST_NAMES = ['query_motion_blur.jsonl', 'query_low_light.jsonl', 'query_gaussian_noise.jsonl']
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
WANDB_PROJECT = 'regram-condition-shift'
WANDB_MODE = 'online'
SHOW_RUNNER_OUTPUT = True
RUNNER_OUTPUT_TAIL_LINES = 40
RUN_ONLY_READY_BASELINES = True
STOP_ON_FAILURE = False
TOP_K_WORST_SHIFTS = 10

RAW_LOCO_ROOT = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection'
UNIVAD_CAPTION_DATA_ROOT = REPO_ROOT / 'data' / 'mvtec_loco_caption'
UNIVAD_SETUP_FLAGS = default_univad_setup_flags()

BASELINE_SPECS = build_baseline_specs(
    repo_root=REPO_ROOT,
    exp_root=EXP_ROOT,
    raw_loco_root=RAW_LOCO_ROOT,
    univad_caption_data_root=UNIVAD_CAPTION_DATA_ROOT,
    default_patchcore_device=DEFAULT_PATCHCORE_DEVICE,
)
validate_baseline_names(DASHBOARD_BASELINES, specs=BASELINE_SPECS, label='dashboard')
REQUESTED_BASELINES = resolve_requested_baselines(
    RUN_MODE,
    RUN_BASELINE,
    SEQUENTIAL_BASELINES,
    specs=BASELINE_SPECS,
)
MANIFEST_NAMES = discover_manifest_names(
    manifest_roots=MANIFEST_ROOT_CANDIDATES,
    configured_names=MANIFEST_NAMES,
    auto_discover=AUTO_DISCOVER_MANIFESTS,
    excluded_names=EXCLUDED_MANIFEST_NAMES,
)
manifest_paths = resolve_manifest_paths(MANIFEST_NAMES, MANIFEST_ROOT_CANDIDATES)
requested_run_configs = build_requested_run_configs(
    active_baselines=REQUESTED_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
)
dashboard_run_configs = build_requested_run_configs(
    active_baselines=DASHBOARD_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
)
baseline_status_df = evaluate_baseline_readiness(REQUESTED_BASELINES, specs=BASELINE_SPECS, categories=CATEGORIES)
run_configs = list(requested_run_configs)
skipped_run_configs = []

display_experiment_preset(
    run_mode=RUN_MODE,
    requested_baselines=REQUESTED_BASELINES,
    dashboard_baselines=DASHBOARD_BASELINES,
    categories=CATEGORIES,
    manifest_names=MANIFEST_NAMES,
    wandb_mode=WANDB_MODE,
    stop_on_failure=STOP_ON_FAILURE,
    univad_flags=UNIVAD_SETUP_FLAGS,
    specs=BASELINE_SPECS,
    requested_run_configs=requested_run_configs,
)


In [ ]:
display_title('Current Run Matrix', 'Single mode executes one baseline. Sequential mode executes the listed baselines in order and keeps a shared comparison dashboard.')
command_preview_df = pd.DataFrame([
    {
        'baseline': config['baseline'],
        'category': config['category'],
        'manifest_count': len(config['manifest_paths']),
        'device': config['device'],
        'summary_name': config['summary_path'].name,
        'log_name': config['log_path'].name,
        'runner_cmd': ' '.join(config['runner_cmd']),
    }
    for config in requested_run_configs
])
display(command_preview_df)


In [ ]:
display_title(
    'Baseline Setup & Readiness',
    'Setup/check external repos, runtime dependencies, checkpoints, datasets, and masks before execution.',
)
setup_result = run_baseline_setup(
    requested_baselines=REQUESTED_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    raw_loco_root=RAW_LOCO_ROOT,
    settings=UNIVAD_SETUP_FLAGS,
    requested_run_configs=requested_run_configs,
    run_only_ready_baselines=RUN_ONLY_READY_BASELINES,
)
setup_df = setup_result['setup_df']
baseline_status_df = setup_result['baseline_status_df']
run_configs = setup_result['run_configs']
skipped_run_configs = setup_result['skipped_run_configs']

display(setup_df)
if 'UniVAD' in REQUESTED_BASELINES:
    for title, key in [
        ('UniVAD Dataset Status', 'univad_dataset_status_df'),
        ('UniVAD Local Dependency Status', 'univad_local_dependency_status_df'),
        ('UniVAD Checkpoint Status', 'univad_checkpoint_status_df'),
        ('UniVAD Grounding Mask Status', 'univad_mask_status_df'),
    ]:
        display_title(title)
        display(setup_result[key])

display_title('Baseline Readiness')
display(baseline_status_df)
if skipped_run_configs:
    display_title('Skipped Runs')
    display(pd.DataFrame([
        {
            'baseline': config['baseline'],
            'category': config['category'],
            'summary_path': str(config['summary_path']),
            'log_path': str(config['log_path']),
        }
        for config in skipped_run_configs
    ]))

display_title('Execution Queue', f'Prepared `{len(run_configs)}` runnable jobs.')
display_run_plan(run_configs)


In [ ]:
run_history = run_execution_queue(
    run_configs=run_configs,
    repo_root=REPO_ROOT,
    show_runner_output=SHOW_RUNNER_OUTPUT,
    runner_output_tail_lines=RUNNER_OUTPUT_TAIL_LINES,
    stop_on_failure=STOP_ON_FAILURE,
)


## Result Dashboard

이 아래 셀들은 runner가 남긴 artifact를 다시 읽어서, clean 기준 성능과 shift 취약성을 실험 리포트처럼 정리한다.

읽는 대상:

- manifest-shift `summary.json`
- clean-eval artifact
- identity reproduction / manifest validation artifact

보는 관점:

- clean data에 대한 재현성과 기준 성능
- shift data에서 severity별, family별 붕괴 패턴
- PatchCore와 UniVAD의 상대 우위와 약점 차이

중요:

- dashboard는 `DASHBOARD_BASELINES`에 포함된 누적 결과를 함께 읽는다.
- 따라서 오늘 `PatchCore`만 돌리고, 나중에 `UniVAD`를 돌려도 같은 표와 heatmap에서 같이 비교할 수 있다.


In [ ]:
summary_sources = dashboard_run_configs if 'dashboard_run_configs' in globals() else requested_run_configs
dashboard_frames = load_dashboard_frames(
    summary_sources=summary_sources,
    dashboard_baselines=DASHBOARD_BASELINES,
    categories=CATEGORIES,
    report_root=REPORT_ROOT,
    severity_order=SEVERITY_ORDER,
    top_k_worst_shifts=TOP_K_WORST_SHIFTS,
)
globals().update(dashboard_frames)


In [ ]:
display_dashboard_tables(
    dashboard_frames,
    run_history=run_history,
    dashboard_baselines=DASHBOARD_BASELINES,
    top_k=TOP_K_WORST_SHIFTS,
)


In [ ]:
display_wandb_panel_recommendations()


In [ ]:
render_visual_dashboard(dashboard_frames, dashboard_baselines=DASHBOARD_BASELINES)
